# Benchmarks Ablations and Robustness

**Question:** Which added components actually improve performance?

**Prerequisites:** install the project with notebook extras; run from the repository root. This notebook is self-contained in bounded demo mode.

## Theory and assumptions

Ablations remove one contribution at a time while holding data access and tuning budgets fixed. Paired comparisons use the same independent groups. Group bootstrap confidence intervals resample geometries or patients, not dependent pixels. Report computational cost alongside accuracy. Robustness perturbations are stress tests; they do not recreate every scanner domain shift.

In [ ]:
from pathlib import Path
import os,json
import numpy as np
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'configs').exists():ROOT=ROOT.parent
mode='demo'
data_root=ROOT/'data/raw'
config_path=ROOT/'configs/demo.yaml'
seed=42
max_samples=8
device='cpu'
output_dir=ROOT/'reports/notebooks'/'16_Benchmarks_Ablations_and_Robustness'
output_dir.mkdir(parents=True,exist_ok=True)
print('Mode:',mode,'| Data origin: synthetic_fixture')


## Experiment

Inputs: seeded synthetic fixtures or explicitly identified educational probability tables. Outputs: printed numerical evidence, saved run artifacts where applicable, and plots. Real-data-only claims remain unavailable until source archives and metadata are verified.

In [ ]:
from elastograde.data.synthetic import make_pair
from elastograde.evaluation.robustness import corrupt
from elastograde.inference.pipeline import infer
from elastograde.evaluation.metrics import displacement_metrics
from elastograde.evaluation.grouped_statistics import grouped_bootstrap
rows=[]
for i in range(4):
 s=make_pair(seed+i)
 for severity in [0,.2,.5]:
  c=corrupt(s,'noise',severity,seed+i);r=infer(c)
  rows.append({'group':i,'severity':severity,**displacement_metrics(r['flow'],s.displacement,r['valid'])})
for severity in [0,.2,.5]:
 vals=[r['endpoint_error'] for r in rows if r['severity']==severity]
 print(severity,grouped_bootstrap(vals,list(range(4)),repeats=200))
plt.plot([r['severity'] for r in rows],[r['endpoint_error'] for r in rows],'o');plt.title('synthetic_fixture: NCC noise stress');plt.xlabel('Noise fraction');plt.ylabel('EPE pixels');plt.show()

## Interpretation and failure cases

Only four toy groups are used here. Confidence intervals are illustrative and small-sample uncertainty is substantial. Real benchmark runs are not implied.

## Check your understanding

Why compare models on identical groups? Answer: paired differences reduce variation caused by unequal test difficulty and make the scientific comparison interpretable.

## Evidence statement

This execution demonstrates the computations shown above. It does not establish clinical accuracy, generalization to patients, or state-of-the-art performance. Record the environment and split protocol before a full experiment.